In [1]:

import os
xx = !cat ~/openai_api_key
os.environ['OPENAI_API_KEY'] = xx[0]
del xx

from reevo.problems.bpp_offline_aco.eval_jk import BPPoffACOEvaluator, promptdetails as BPPoffACO_prompts
from reevo.problems.cvrp_aco.eval_jk import CVRPACOEvaluator, promptdetails as CVRPACO_prompts
from reevo.problems.mkp_aco.eval_jk import MKPACOEvaluator, promptdetails as MKPACO_prompts
from reevo.problems.op_aco.eval_jk import OPACOEvaluator, promptdetails as OPACO_prompts
from reevo.problems.tsp_aco.eval_jk import TSPACOEvaluator, promptdetails as TSPACO_prompts

tasks = dict(
    bpp_offline_aco=(BPPoffACOEvaluator(), BPPoffACO_prompts),
    cvrp_aco=(CVRPACOEvaluator(), CVRPACO_prompts),
    mkp_aco=(MKPACOEvaluator(), MKPACO_prompts),
    op_aco=(OPACOEvaluator(), OPACO_prompts),
    tsp_aco=(TSPACOEvaluator(), TSPACO_prompts)
)

from algorithms.eoh import eoh, engine as eoh_engine
from reallm import ProblemSpec, Evaluation
import re

defining engine...
defining engine...
setting up engine
defining prompts


In [3]:
%%time

probresults = dict()
for task_name, (task_evaluator, task_prompts) in tasks.items():

    if task_name in ['bpp_offline_aco', 'mkp_aco', 'cvrp_aco', 'tsp_aco']:
        continue
    
    print(f'---==== TRYING {task_name} ====---')
    eoh_engine.start_trace()

    def fitness(func):
        try:
            return Evaluation(
                fitness = task_evaluator.evaluate(func),
                errors = None
            )
        except Exception as e:
            return Evaluation(
                fitness = float('inf'),
                errors = str(e)
            )

    
    res = eoh(
        num_generations=1,
        problem_spec=ProblemSpec(
            description=task_prompts["func_desc"],
            template=task_prompts["func_signature"] + '\n\tpass',
            entrypoint="heuristic",
            evaluator=fitness
        )
    )

    tg, tl = eoh_engine.dump_trace()
    probresults[task_name] = {
        'bestsoln': res,
        'tracegraph': tg,
        'tracelist': tl
    }
    

---==== TRYING op_aco ====---
Building initial 3 solns..
	building generation 0
CPU times: user 1min 51s, sys: 793 ms, total: 1min 52s
Wall time: 1min 25s


In [4]:
[print(i,j['bestsoln'].fitness) for i,j in probresults.items()]

op_aco 6.232000000000001


[None]

In [4]:
print(probresults['op_aco']['bestsoln'])


solution:

import numpy as np

def heuristic(prize: np.ndarray, distance: np.ndarray, maxlen: float) -> np.ndarray:
    n = prize.shape[0]
    heuristics = np.zeros((n, n))
    
    for i in range(n):
        for j in range(n):
            if i != j and distance[i][j] <= maxlen:
                heuristics[i][j] = prize[j] / distance[i][j]  # Promising edges based on prize/distance ratio

    return heuristics


thought: The algorithm calculates a heuristic score for including edges between nodes based on their associated prizes and the distance constraints, helping to prioritize edges that maximize rewards while respecting travel limitations.

evaluation: Evaluation(fitness=inf, errors='OPACOEvaluator._solve() takes 2 positional arguments but 3 were given')



In [ ]:

tasks = dict(
    bpp_offline_aco=(BPPoffACOEvaluator(), BPPoffACO_prompts),
    cvrp_aco=(CVRPACOEvaluator(), CVRPACO_prompts),
    mkp_aco=(MKPACOEvaluator(), MKPACO_prompts),
    op_aco=(OPACOEvaluator(), OPACO_prompts),
    tsp_aco=(TSPACOEvaluator(), TSPACO_prompts)
)

In [8]:

import numpy as np

def heuristic(prize: np.ndarray, distance: np.ndarray, maxlen: float) -> np.ndarray:
    n = len(prize)
    heuristics = np.zeros((n, n))
    
    for i in range(n):
        for j in range(n):
            if i != j and distance[i][j] <= maxlen:
                heuristics[i][j] = prize[j] / distance[i][j]
    
    return heuristics


OPACOEvaluator().evaluate(heuristic)

AttributeError: 'str' object has no attribute 'prize'

In [7]:
tl

In [6]:
type(tg)

NoneType

In [7]:
!pwd

/root/reallm
